# Üye 1 — AST + Graf + Pipeline
YZM358 Doğal Dil İşleme Projesi

In [5]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q numpy==1.26.4
!pip install -q tree-sitter==0.21.3 tree-sitter-python==0.21.0 tree-sitter-java==0.21.0
!pip install -q node2vec==0.4.6
!pip install -q transformers==4.41.0 sentencepiece networkx gradio
!pip install -q faiss-cpu

print('✅ Kurulum tamamlandı!')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Kurulum tamamlandı!


In [6]:
import tree_sitter_python as tspython
import tree_sitter_java as tsjava
from tree_sitter import Language, Parser
import networkx as nx
import numpy as np
from node2vec import Node2Vec
import torch
from transformers import RobertaTokenizer, T5ForConditionalGeneration

print('✅ Kütüphaneler hazır!')

✅ Kütüphaneler hazır!


In [7]:
PY_LANGUAGE = Language(tspython.language(), 'python')
JAVA_LANGUAGE = Language(tsjava.language(), 'java')

def get_ast(code, lang):
    parser = Parser()
    if lang == 'python':
        parser.set_language(PY_LANGUAGE)
    elif lang == 'java':
        parser.set_language(JAVA_LANGUAGE)
    else:
        raise ValueError(f"Desteklenmeyen dil: {lang}. 'python' veya 'java' kullanın.")
    return parser.parse(bytes(code, 'utf-8')).root_node

print('✅ Parser hazır!')

✅ Parser hazır!


## 🔧 Düzeltme 1: `node_id` mutable default argument hatası giderildi
## 🔧 Düzeltme 3: Node2Vec cache eklendi (aynı graf için tekrar hesaplama yapılmıyor)

In [8]:
# DÜZELTME 1: node_id=[0] yerine node_id=None kullanıldı
def ast_to_graph(node, graph=None, parent_id=None, node_id=None):
    """AST'yi NetworkX yönlü grafına dönüştürür."""
    if node_id is None:      # Her çağrıda yeni liste oluştur
        node_id = [0]
    if graph is None:
        graph = nx.DiGraph()
        node_id[0] = 0
    current_id = node_id[0]
    node_id[0] += 1
    graph.add_node(current_id, type=node.type,
                   text=node.text.decode('utf-8') if node.text else '')
    if parent_id is not None:
        graph.add_edge(parent_id, current_id)
    for child in node.children:
        ast_to_graph(child, graph, current_id, node_id)
    return graph


# DÜZELTME 3: Node2Vec her seferinde yeniden fit edilmiyor
_encoding_cache = {}

def graph_to_encoding(graph, dimensions=64):
    """Node2Vec ile graf yapısını vektöre dönüştürür. Sonuçlar cache'lenir."""
    if graph.number_of_nodes() == 0:
        return np.zeros(dimensions)
    cache_key = str(sorted(graph.edges()))
    if cache_key in _encoding_cache:
        return _encoding_cache[cache_key]
    n2v = Node2Vec(graph, dimensions=dimensions, walk_length=10,
                   num_walks=20, workers=1, quiet=True)
    m = n2v.fit(window=5, min_count=1)
    result = np.mean([m.wv[str(n)] for n in graph.nodes()], axis=0)
    _encoding_cache[cache_key] = result
    return result

print('✅ Graf ve Encoding hazır! (node_id hatası düzeltildi, cache eklendi)')

✅ Graf ve Encoding hazır! (node_id hatası düzeltildi, cache eklendi)


In [9]:
model_path = '/content/drive/MyDrive/CodeReviewBot/model_ciktisi/final_model'
device = 'cuda' if torch.cuda.is_available() else 'cpu'

tokenizer = RobertaTokenizer.from_pretrained(model_path)
model = T5ForConditionalGeneration.from_pretrained(model_path).to(device)
model.eval()

print(f'✅ Model yüklendi! Cihaz: {device.upper()}')

✅ Model yüklendi! Cihaz: CPU


pipeline_content = '''import sys, re, pickle, numpy as np, networkx as nx, torch, faiss
import tree_sitter_python as tspython
import tree_sitter_java as tsjava
from tree_sitter import Language, Parser
from node2vec import Node2Vec
from transformers import RobertaTokenizer, T5ForConditionalGeneration

DRIVE = "/content/drive/MyDrive/CodeReviewBot"
PY_LANGUAGE = Language(tspython.language(), "python")
JAVA_LANGUAGE = Language(tsjava.language(), "java")
device = "cuda" if torch.cuda.is_available() else "cpu"

# Review model
tokenizer = RobertaTokenizer.from_pretrained(f"{DRIVE}/model_ciktisi/final_model")
model = T5ForConditionalGeneration.from_pretrained(f"{DRIVE}/model_ciktisi/final_model").to(device)
model.eval()
print("[Pipeline] Review model yuklendi!")

# FAISS
faiss_index = faiss.read_index(f"{DRIVE}/model_ciktisi/faiss_index.bin")
with open(f"{DRIVE}/model_ciktisi/corpus_data.pkl", "rb") as f:
    corpus_data = pickle.load(f)
print(f"[Pipeline] FAISS yuklendi! ({faiss_index.ntotal} vektor)")

# Repair model
repair_tokenizer = RobertaTokenizer.from_pretrained(f"{DRIVE}/model_ciktisi/repair_model/")
repair_model = T5ForConditionalGeneration.from_pretrained(f"{DRIVE}/model_ciktisi/repair_model/").to(device)
repair_model.eval()
print("[Pipeline] Repair model yuklendi!")


def get_ast(code, lang):
    parser = Parser()
    if lang == "python":
        parser.set_language(PY_LANGUAGE)
    elif lang == "java":
        parser.set_language(JAVA_LANGUAGE)
    else:
        raise ValueError(f"Desteklenmeyen dil: {lang}")
    return parser.parse(bytes(code, "utf-8")).root_node


# DUZELTME 1: node_id=None (mutable default arg hatasi giderildi)
def ast_to_graph(node, graph=None, parent_id=None, node_id=None):
    if node_id is None:
        node_id = [0]
    if graph is None:
        graph = nx.DiGraph()
        node_id[0] = 0
    current_id = node_id[0]
    node_id[0] += 1
    graph.add_node(current_id, type=node.type,
                   text=node.text.decode("utf-8") if node.text else "")
    if parent_id is not None:
        graph.add_edge(parent_id, current_id)
    for child in node.children:
        ast_to_graph(child, graph, current_id, node_id)
    return graph


# DUZELTME 3: Encoding cache
_encoding_cache = {}
def graph_to_encoding(graph, dimensions=64):
    if graph.number_of_nodes() == 0:
        return np.zeros(dimensions)
    cache_key = str(sorted(graph.edges()))
    if cache_key in _encoding_cache:
        return _encoding_cache[cache_key]
    n2v = Node2Vec(graph, dimensions=dimensions, walk_length=10,
                   num_walks=20, workers=1, quiet=True)
    m = n2v.fit(window=5, min_count=1)
    result = np.mean([m.wv[str(n)] for n in graph.nodes()], axis=0)
    _encoding_cache[cache_key] = result
    return result


def rag_ara(query_code, k=2):
    enc_in = tokenizer(query_code, return_tensors="pt",
                       truncation=True, max_length=512).to(device)
    with torch.no_grad():
        emb = model.encoder(**enc_in).last_hidden_state.mean(dim=1)
        emb = emb.cpu().numpy().astype("float32")
    _, idxs = faiss_index.search(emb, k)
    return [corpus_data[i] for i in idxs[0] if i != -1]


def kategori_belirle(review):
    r = review.lower()
    if any(k in r for k in ["security","injection","vulnerability","unsafe","attack","password","auth"]):
        return "Guvenlik"
    elif any(k in r for k in ["performance","slow","inefficient","loop","complexity","optimize","memory"]):
        return "Performans"
    elif any(k in r for k in ["naming","readability","style","format","comment","docstring","variable"]):
        return "Okunabilirlik"
    return "Genel"


# DUZELTME 2: [OLD] artik bos degil
def code_review(code, lang="python", verbose=False):
    try:
        ast = get_ast(code, lang)
        graf = ast_to_graph(ast)
        encoding = graph_to_encoding(graf)
        node_tipleri = list(set(nx.get_node_attributes(graf, "type").values()))[:5]
        graf_bilgisi = (f"[GRAPH ANALYSIS] Nodes: {graf.number_of_nodes()}, "
                        f"Edges: {graf.number_of_edges()}, "
                        f"Node types: {', '.join(node_tipleri)}")
    except Exception as e:
        print(f"[Uyari] Graf analizi basarisiz: {e}")
        graf = nx.DiGraph()
        graf_bilgisi = "[GRAPH ANALYSIS] Graf olusturulamadi."

    rag_ornekler = rag_ara(code, k=2)
    rag_metni = ""
    if rag_ornekler:
        rag_metni = "Similar reviews:\n" + "\n".join([f"- {str(o.get('msg', ''))[:80]}" for o in rag_ornekler])

    if verbose:
        print("\nRAG Ornekleri:")
        for i, o in enumerate(rag_ornekler):
            print(f"  [{i+1}] {str(o.get('msg', ''))[:100]}")

    yarim = len(code) // 2
    prompt = (
        f"Review this code change:\n"
        f"{graf_bilgisi}\n"
        f"{rag_metni}\n"
        f"[OLD]: {code[:yarim]}\n"
        f"[NEW]: {code[:400]}"
    )

    inputs = tokenizer(prompt, return_tensors="pt", max_length=512, truncation=True).to(device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_length=128, min_length=5,
                                  num_beams=4, no_repeat_ngram_size=3, early_stopping=True)
    review = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return {
        "review": review,
        "kategori": kategori_belirle(review),
        "graph_nodes": graf.number_of_nodes(),
        "graph_edges": graf.number_of_edges(),
        "rag_ornekler": [str(o.get("msg","")) for o in rag_ornekler]
    }


def code_repair_model_fn(code, lang="python"):
    prompt = f"fix {lang}: {code[:400]}"
    inputs = repair_tokenizer(prompt, return_tensors="pt",
                              max_length=256, truncation=True).to(device)
    with torch.no_grad():
        outputs = repair_model.generate(**inputs, max_length=256, num_beams=4, early_stopping=True)
    return repair_tokenizer.decode(outputs[0], skip_special_tokens=True)


def code_repair_kural(code, lang="python"):
    duzeltmeler = []
    yeni_kod = code

    # SQL Injection
    if "execute(" in code and ('" +' in code or "' +" in code):
        yeni_kod = re.sub(
            r'\.execute\(["\'](.+?)["\'] \+ (\w+)\)',
            r'.execute("\1?", (\2,))',
            yeni_kod
        )
        duzeltmeler.append("SQL injection duzeltildi — parameterized query kullanildi")

    # Dosya kapatilmiyor
    if "open(" in code and "close()" not in code and "with open" not in code:
        satirlar = yeni_kod.split('\n')
        yeni_satirlar = []
        for satir in satirlar:
            if satir.strip().startswith('return') and 'dosya' not in satir:
                yeni_satirlar.append('    dosya.close()')
            yeni_satirlar.append(satir)
        yeni_kod = '\n'.join(yeni_satirlar)
        duzeltmeler.append("Dosya kapatma eklendi")

    # Sabit sifre
    if any(k in code for k in ['"1234"', "'1234'", '"admin"', "'admin'"]):
        yeni_kod = "# Sabit sifre! Environment variable kullanin.\n" + yeni_kod
        duzeltmeler.append("Sabit sifre tespit edildi")

    if not duzeltmeler:
        return code, ["Bilinen hata kalibi bulunamadi"]
    return yeni_kod, duzeltmeler


def code_repair(code, lang="python"):
    if lang == "java":
        return code_repair_model_fn(code, lang), ["Java repair modeli kullanildi"]
    yeni_kod, mesajlar = code_repair_kural(code, lang)
    if "Bilinen hata kalibi bulunamadi" in mesajlar:
        ai_duzeltme = code_repair_model_fn(code, lang)
        return ai_duzeltme, ["Yapay Zeka (CodeT5+) ile duzeltildi"]
    return yeni_kod, mesajlar


print("Pipeline hazir! Tum duzeltmeler uygulanmis.")
'''

with open('/content/drive/MyDrive/CodeReviewBot/utils/pipeline.py', 'w', encoding='utf-8') as f:
    f.write(pipeline_content)

print('✅ pipeline.py başarıyla güncellendi! (3 düzeltme uygulandı)')

In [10]:
def code_review(code, lang):
    """Tam pipeline: AST/Graf analizi + CodeT5+ yorum üretimi."""
    ast  = get_ast(code, lang)
    graph = ast_to_graph(ast)
    encoding = graph_to_encoding(graph)

    # DÜZELTME 2: [OLD] artık boş değil — kodun ilk yarısı eski, tamamı yeni olarak gösteriliyor
    yarim = len(code) // 2
    prompt = (
        f"Review this code change:\n"
        f"[OLD]: {code[:yarim]}\n"
        f"[NEW]: {code[:500]}"
    )
    inputs = tokenizer(prompt, return_tensors='pt',
                       max_length=512, truncation=True).to(device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=128,
            min_length=5,
            num_beams=4,
            no_repeat_ngram_size=3,
            early_stopping=True
        )
    review = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return {
        'review': review,
        'graph_nodes': graph.number_of_nodes(),
        'encoding_shape': encoding.shape
    }

print('✅ code_review fonksiyonu hazır!')

✅ code_review fonksiyonu hazır!


In [11]:
test_kodu = """
def kullanici_sil(db, user_id):
    db.execute("DELETE FROM users WHERE id=" + user_id)
"""

sonuc = code_review(test_kodu, 'python')
print('Review:', sonuc['review'])
print('Graf düğüm sayısı:', sonuc['graph_nodes'])
print('Encoding boyutu:', sonuc['encoding_shape'])

Review: Why remove this?
Graf düğüm sayısı: 28
Encoding boyutu: (64,)


## Pipeline.py — Tam Entegrasyon (Düzeltilmiş)
Tüm düzeltmeler bu dosyaya da yansıtıldı.

In [16]:
import sys, re, pickle
import numpy as np
import networkx as nx
import torch
import faiss
import tree_sitter_python as tspython
import tree_sitter_java as tsjava
from tree_sitter import Language, Parser
from node2vec import Node2Vec
from transformers import RobertaTokenizer, T5ForConditionalGeneration

DRIVE = "/content/drive/MyDrive/CodeReviewBot"
PY_LANGUAGE   = Language(tspython.language(), "python")
JAVA_LANGUAGE = Language(tsjava.language(), "java")
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = RobertaTokenizer.from_pretrained(f"{DRIVE}/model_ciktisi/final_model")
model     = T5ForConditionalGeneration.from_pretrained(f"{DRIVE}/model_ciktisi/final_model").to(device)
model.eval()
print("[Pipeline] Review model yuklendi!")

faiss_index = faiss.read_index(f"{DRIVE}/model_ciktisi/faiss_index.bin")
with open(f"{DRIVE}/model_ciktisi/corpus_data.pkl", "rb") as f:
    corpus_data = pickle.load(f)
print(f"[Pipeline] FAISS yuklendi! ({faiss_index.ntotal} vektor)")

repair_tokenizer = RobertaTokenizer.from_pretrained(f"{DRIVE}/model_ciktisi/repair_model/")
repair_model     = T5ForConditionalGeneration.from_pretrained(f"{DRIVE}/model_ciktisi/repair_model/").to(device)
repair_model.eval()
print("[Pipeline] Repair model yuklendi!")


def get_ast(code, lang):
    parser = Parser()
    if lang == "python":
        parser.set_language(PY_LANGUAGE)
    elif lang == "java":
        parser.set_language(JAVA_LANGUAGE)
    else:
        raise ValueError(f"Desteklenmeyen dil: {lang}")
    return parser.parse(bytes(code, "utf-8")).root_node


def ast_to_graph(node, graph=None, parent_id=None, node_id=None):
    if node_id is None:
        node_id = [0]
    if graph is None:
        graph = nx.DiGraph()
        node_id[0] = 0
    current_id = node_id[0]
    node_id[0] += 1
    graph.add_node(current_id, type=node.type,
                   text=node.text.decode("utf-8") if node.text else "")
    if parent_id is not None:
        graph.add_edge(parent_id, current_id)
    for child in node.children:
        ast_to_graph(child, graph, current_id, node_id)
    return graph


_encoding_cache = {}

def graph_to_encoding(graph, dimensions=64):
    if graph.number_of_nodes() == 0:
        return np.zeros(dimensions)
    cache_key = str(sorted(graph.edges()))
    if cache_key in _encoding_cache:
        return _encoding_cache[cache_key]
    n2v = Node2Vec(graph, dimensions=dimensions, walk_length=10,
                   num_walks=20, workers=1, quiet=True)
    m = n2v.fit(window=5, min_count=1)
    result = np.mean([m.wv[str(n)] for n in graph.nodes()], axis=0)
    _encoding_cache[cache_key] = result
    return result


def rag_ara(query_code, k=2):
    enc_in = tokenizer(query_code, return_tensors="pt",
                       truncation=True, max_length=512).to(device)
    with torch.no_grad():
        emb = model.encoder(**enc_in).last_hidden_state.mean(dim=1)
        emb = emb.cpu().numpy().astype("float32")
    _, idxs = faiss_index.search(emb, k)
    return [corpus_data[i] for i in idxs[0] if i != -1]


def kategori_belirle(review):
    r = review.lower()
    if any(k in r for k in ["security", "injection", "vulnerability", "unsafe", "attack", "password", "auth"]):
        return "Guvenlik"
    elif any(k in r for k in ["performance", "slow", "inefficient", "loop", "complexity", "optimize", "memory"]):
        return "Performans"
    elif any(k in r for k in ["naming", "readability", "style", "format", "comment", "docstring", "variable"]):
        return "Okunabilirlik"
    return "Genel"


def code_review(code, lang="python", verbose=False):
    try:
        ast  = get_ast(code, lang)
        graf = ast_to_graph(ast)
        encoding = graph_to_encoding(graf)
        node_tipleri = list(set(nx.get_node_attributes(graf, "type").values()))[:5]
        graf_bilgisi = (
            f"[GRAPH ANALYSIS] Nodes: {graf.number_of_nodes()}, "
            f"Edges: {graf.number_of_edges()}, "
            f"Node types: {', '.join(node_tipleri)}"
        )
    except Exception as e:
        print(f"[Uyari] Graf analizi basarisiz: {e}")
        graf = nx.DiGraph()
        graf_bilgisi = "[GRAPH ANALYSIS] Graf olusturulamadi."

    rag_ornekler = rag_ara(code, k=2)
    rag_satirlar = ["- " + str(o.get("msg", ""))[:80] for o in rag_ornekler]
    rag_metni = ""
    if rag_satirlar:
        rag_metni = "Similar reviews:\n" + "\n".join(rag_satirlar)

    if verbose:
        print("\nRAG Ornekleri:")
        for i, o in enumerate(rag_ornekler):
            print(f"  [{i+1}] {str(o.get('msg', ''))[:100]}")

    yarim = len(code) // 2
    prompt = "\n".join([
        "Review this code change:",
        graf_bilgisi,
        rag_metni,
        "[OLD]: " + code[:yarim],
        "[NEW]: " + code[:400],
    ])

    inputs = tokenizer(prompt, return_tensors="pt", max_length=512, truncation=True).to(device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=128,
            min_length=5,
            num_beams=4,
            no_repeat_ngram_size=3,
            early_stopping=True
        )
    review = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return {
        "review"      : review,
        "kategori"    : kategori_belirle(review),
        "graph_nodes" : graf.number_of_nodes(),
        "graph_edges" : graf.number_of_edges(),
        "rag_ornekler": [str(o.get("msg", "")) for o in rag_ornekler]
    }


def code_repair_model_fn(code, lang="python"):
    prompt = f"fix {lang}: {code[:400]}"
    inputs = repair_tokenizer(prompt, return_tensors="pt",
                              max_length=256, truncation=True).to(device)
    with torch.no_grad():
        outputs = repair_model.generate(**inputs, max_length=256,
                                        num_beams=4, early_stopping=True)
    return repair_tokenizer.decode(outputs[0], skip_special_tokens=True)


def code_repair_kural(code, lang="python"):
    duzeltmeler = []
    yeni_kod = code

    if "execute(" in code and ('" +' in code or "' +" in code):
        yeni_kod = re.sub(
            r'\.execute\(["\'](.+?)["\'] \+ (\w+)\)',
            r'.execute("\1?", (\2,))',
            yeni_kod
        )
        duzeltmeler.append("SQL injection duzeltildi")

    if "open(" in code and "close()" not in code and "with open" not in code:
        satirlar = yeni_kod.split("\n")
        yeni_satirlar = []
        for satir in satirlar:
            if satir.strip().startswith("return") and "dosya" not in satir:
                yeni_satirlar.append("    dosya.close()")
            yeni_satirlar.append(satir)
        yeni_kod = "\n".join(yeni_satirlar)
        duzeltmeler.append("Dosya kapatma eklendi")

    if any(k in code for k in ['"1234"', "'1234'", '"admin"', "'admin'"]):
        yeni_kod = "# Sabit sifre! Environment variable kullanin.\n" + yeni_kod
        duzeltmeler.append("Sabit sifre tespit edildi")

    if not duzeltmeler:
        return code, ["Bilinen hata kalibi bulunamadi"]
    return yeni_kod, duzeltmeler


def code_repair(code, lang="python"):
    if lang == "java":
        return code_repair_model_fn(code, lang), ["Java repair modeli kullanildi"]
    yeni_kod, mesajlar = code_repair_kural(code, lang)
    if "Bilinen hata kalibi bulunamadi" in mesajlar:
        ai_duzeltme = code_repair_model_fn(code, lang)
        return ai_duzeltme, ["Yapay Zeka (CodeT5+) ile duzeltildi"]
    return yeni_kod, mesajlar


print("✅ Pipeline hazir! Tum duzeltmeler uygulanmis.")

[Pipeline] Review model yuklendi!
[Pipeline] FAISS yuklendi! (2000 vektor)
[Pipeline] Repair model yuklendi!
✅ Pipeline hazir! Tum duzeltmeler uygulanmis.


In [15]:
exec(open('/content/drive/MyDrive/CodeReviewBot/utils/pipeline.py').read())
print('✅ Pipeline hazır!')

SyntaxError: unterminated string literal (detected at line 130) (<string>, line 130)

In [17]:
sonuc = code_review(
    "def kullanici_sil(db, user_id):\n    db.execute('DELETE FROM users WHERE id=' + user_id)",
    "python",
    verbose=True
)
print('Review:', sonuc['review'])
print('Kategori:', sonuc['kategori'])
print('RAG örnekleri:', sonuc['rag_ornekler'])


RAG Ornekleri:
  [1] I think in the new APIs we could start passing the country code as mutation parameter since we wante
  [2] inside `_shared/utils` there is a method for parsing connection string. Let's use that.
Review: Why did you remove this?
Kategori: Genel
RAG örnekleri: ["I think in the new APIs we could start passing the country code as mutation parameter since we wanted to stop relying on geolocalization (since it won't work for people using VPN).", "inside `_shared/utils` there is a method for parsing connection string. Let's use that."]


In [18]:
REPAIR_YOLU = '/content/drive/MyDrive/CodeReviewBot/model_ciktisi/repair_model/'

repair_tokenizer = RobertaTokenizer.from_pretrained(REPAIR_YOLU)
repair_model = T5ForConditionalGeneration.from_pretrained(REPAIR_YOLU).to(device)
repair_model.eval()

print(f'✅ Repair model yüklendi!')

✅ Repair model yüklendi!


In [19]:
# Eğer repair_model klasöründe tokenizer yoksa buradan kaydet
from transformers import RobertaTokenizer

tokenizer_base = RobertaTokenizer.from_pretrained('Salesforce/codet5-base')
tokenizer_base.save_pretrained('/content/drive/MyDrive/CodeReviewBot/model_ciktisi/repair_model/')
print('✅ Tokenizer kaydedildi!')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

✅ Tokenizer kaydedildi!


In [20]:
test_py = "def kullanici_sil(db, user_id):\n    db.execute('DELETE FROM users WHERE id=' + user_id)"
test_java = 'public void sil(String id) { db.execute("DELETE FROM users WHERE id=" + id); }'

duzeltilmis, aciklamalar = code_repair(test_py, 'python')
print('Python düzeltme:')
print(duzeltilmis)
print('Açıklamalar:', aciklamalar)

print('---')

duzeltilmis_java, _ = code_repair(test_java, 'java')
print('Java düzeltme:', duzeltilmis_java)

Python düzeltme:
def kullanici_sil(db, user_id):
    db.execute("DELETE FROM users WHERE id=?", (user_id,))
Açıklamalar: ['SQL injection duzeltildi']
---
Java düzeltme: public void delete(String id) { db.execute("DELETE FROM users WHERE id=" + id ) ; } 



## Veri Seti Kontrol

In [21]:
import json

with open('/content/drive/MyDrive/CodeReviewBot/data/codereviewer_clean/train_clean.jsonl') as f:
    ornek = json.loads(f.readline())

print('Alanlar:', list(ornek.keys()))

Alanlar: ['input', 'oldf', 'msg', 'y']


In [22]:
with open('/content/drive/MyDrive/CodeReviewBot/data/codereviewer_clean/train_clean.jsonl') as f:
    for i, line in enumerate(f):
        ornek = json.loads(line)
        print(f'--- Örnek {i+1} ---')
        print('oldf:', str(ornek.get('oldf', ''))[:150])
        print('y:', ornek.get('y'))
        print('input:', str(ornek.get('input', ''))[:100])
        print()
        if i == 3:
            break

--- Örnek 1 ---
oldf: //    |  /           |
//    ' /   __| _` | __|  _ \   __|
//    . \  |   (   | |   (   |\__ `
//   _|\_\_|  \__,_|\__|\___/ ____/
//                 
y: 1
input: @@ -595,8 +595,10 @@ namespace Kratos
             array_1d<double, 3> b = ZeroVector(3);
          

--- Örnek 2 ---
oldf: #
# Licensed to the Apache Software Foundation (ASF) under one or more
# contributor license agreements.  See the NOTICE file distributed with
# this 
y: 1
input: @@ -22,8 +22,13 @@
 For internal use only; no backwards-compatibility guarantees.
 """
 
+from __fut

--- Örnek 3 ---
oldf: # frozen_string_literal: true

require 'view/game/part/blocker'
require 'view/game/part/borders'
require 'view/game/part/cities'
require 'view/game/pa
y: 1
input: @@ -25,13 +25,16 @@ module View
       def should_render_revenue?
         revenue = @tile.revenue_t

--- Örnek 4 ---
oldf: /*
 * (C) Copyright 2016-2021 Intel Corporation.
 *
 * SPDX-License-Identifier: BSD-2-Clause-Patent
 */
/**
 * \file


## F1 / Precision / Recall Değerlendirme

In [23]:
from sklearn.metrics import f1_score, precision_score, recall_score
import json, os

test_ornekler = [
    ("def get_user(u):\n    q = 'SELECT * FROM users WHERE name=' + u\n    return db.execute(q)", 'Guvenlik'),
    ("def topla(a, b):\n    return a + b", 'Genel'),
    ("def bol(a, b):\n    return a / b", 'Genel'),
    ("def login(u, p):\n    if u == 'admin' and p == '1234':\n        return True", 'Guvenlik'),
    ("def liste_yaz(liste):\n    for i in range(len(liste)):\n        print(liste[i])", 'Performans'),
    ("def hesapla(n):\n    s = 0\n    for i in range(n):\n        for j in range(n):\n            s += i*j\n    return s", 'Performans'),
    ("def f(x):\n    return x*2", 'Okunabilirlik'),
    ("def veri_al(db, id):\n    return db.execute('SELECT * FROM t WHERE id=' + id)", 'Guvenlik'),
    ("def kontrol(a, b, c):\n    if a > 0:\n        if b > 0:\n            if c > 0:\n                return True", 'Okunabilirlik'),
    ("def isle(l):\n    r = []\n    for i in l:\n        r.append(i*2)\n    return r", 'Performans'),
]

gercek = [t[1] for t in test_ornekler]
tahmin = []

for kod, _ in test_ornekler:
    sonuc = code_review(kod, 'python')
    tahmin.append(sonuc['kategori'])

print('Gerçek:', gercek)
print('Tahmin:', tahmin)

f1 = f1_score(gercek, tahmin, average='weighted', zero_division=0)
precision = precision_score(gercek, tahmin, average='weighted', zero_division=0)
recall = recall_score(gercek, tahmin, average='weighted', zero_division=0)

print(f'F1: {f1:.2f}')
print(f'Precision: {precision:.2f}')
print(f'Recall: {recall:.2f}')

# Kaydet
os.makedirs('/content/drive/MyDrive/CodeReviewBot/outputs/', exist_ok=True)
with open('/content/drive/MyDrive/CodeReviewBot/model_ciktisi/metrikler.json', 'r') as f:
    sonuclar = json.load(f)

sonuclar['f1'] = round(f1, 2)
sonuclar['precision'] = round(precision, 2)
sonuclar['recall'] = round(recall, 2)

with open('/content/drive/MyDrive/CodeReviewBot/model_ciktisi/metrikler.json', 'w') as f:
    json.dump(sonuclar, f, indent=2)

print('✅ F1 kaydedildi!')

Gerçek: ['Guvenlik', 'Genel', 'Genel', 'Guvenlik', 'Performans', 'Performans', 'Okunabilirlik', 'Guvenlik', 'Okunabilirlik', 'Performans']
Tahmin: ['Genel', 'Genel', 'Genel', 'Genel', 'Genel', 'Genel', 'Genel', 'Genel', 'Genel', 'Genel']
F1: 0.07
Precision: 0.04
Recall: 0.20
✅ F1 kaydedildi!
